# Unsuccessful Pass Type Prediction

**Objective**: Build a predictive model to classify pass types (direction × range) for unsuccessful passes, enabling complete MDP construction.

**Challenge**: Unsuccessful passes lack reception coordinates (target location unknown), so we cannot directly compute pass direction/distance like we do for successful passes.

**Solution**: Train an XGBoost classifier on successful passes using contextual features (position, game state, pressure, etc.) to predict pass type, then apply to unsuccessful passes.

---

## Approach

1. **Feature Engineering**: Select variables available for both successful and unsuccessful passes
2. **Model Training**: XGBoost multi-class classifier on successful passes (~272K samples)
3. **Validation**: Evaluate on held-out successful passes
4. **Application**: Predict pass types for unsuccessful passes (~56K samples)
5. **Integration**: Use predictions in MDP transition matrix construction

---

## 1. Setup & Data Loading

In [1]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Custom modules
from src import data_processing as dp

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Imports successful!")

✓ Imports successful!


In [2]:
# Load Premier League 2024 events and extract passes
print("Loading data...")
events = dp.load_premier_league_events()
passes_raw = dp.extract_pass_events(events)

print(f"\n{'='*60}")
print(f"DATA LOADED")
print(f"{'='*60}")
print(f"Total passes: {len(passes_raw):,}")
print(f"  Successful:   {(passes_raw['success']==1).sum():,} ({(passes_raw['success']==1).sum()/len(passes_raw)*100:.1f}%)")
print(f"  Unsuccessful: {(passes_raw['success']==0).sum():,} ({(passes_raw['success']==0).sum()/len(passes_raw)*100:.1f}%)")

Loading data...
Found 378 matches in /home/macaco3001/Projectos/Twelve/twelve-deep-learning/PremierLeague_data/2024/dynamic
  Loaded 50/378 matches...
  Loaded 100/378 matches...
  Loaded 150/378 matches...
  Loaded 200/378 matches...
  Loaded 250/378 matches...
  Loaded 300/378 matches...
  Loaded 350/378 matches...
✅ Loaded 1,811,078 total events from 378 matches
   Event types: 4
   Date range: 1650385 to 2018580
Extracting pass events from 1,811,078 total events...
  Found 329,716 pass events (end_type='pass')
  Removing 1,003 successful passes with missing reception coordinates
  (0.37% of successful passes)
  Pass outcomes:
    Successful:   272,335 (82.8%)
    Unsuccessful: 55,412 (16.9%)
    Offside:         966 ( 0.3%)
✅ Extracted 328,713 pass events

DATA LOADED
Total passes: 328,713
  Successful:   272,335 (82.8%)
  Unsuccessful: 56,378 (17.2%)


## 2. Variable Availability Analysis

### Key Clarifications

**IMPORTANT**: For unsuccessful passes, we do NOT have reception coordinates:
- ❌ `player_targeted_x_reception` / `player_targeted_y_reception` = NULL
- ❌ `pass_distance` = NULL (requires reception coords)
- ❌ `pass_direction` = NULL (requires reception coords)

**What we DO have**:
- ✅ `x_start`, `y_start` = where the passer was when they released the ball
- ✅ `x_end`, `y_end` = where the passer was at the end of their possession (typically same as x_start/y_start for passes, but still useful as features)

**Implication**: We must rely on *contextual* features rather than *geometric* features (distance/angle to reception point).

In [3]:
# Proposed variables for predictive model
proposed_vars = [
    # Coordinates (passer location)
    'x_start', 'y_start', 'x_end', 'y_end',
    # Note: x_end/y_end are typically same as x_start/y_start for passes, but include for completeness
    
    # Spatial context
    'third_start',
    'channel_start',
    'penalty_area_start',
    
    # Game context
    'game_state',
    'team_in_possession_phase_type',
    'team_out_of_possession_phase_type',
    
    # Pressure & support
    'n_teammates_ahead_start',
    'n_opponents_ahead_start',
    'separation_start',
    'organised_defense',  # British spelling!
    
    # Action characteristics
    'one_touch',
    'quick_pass',
    'carry',
    'forward_momentum',
    'is_header',
    'hand_pass',
    
    # Physical metrics
    'speed_avg',
    'high_pass',
]

print("="*80)
print("VARIABLE AVAILABILITY ANALYSIS")
print("="*80)

# Separate successful and unsuccessful passes
successful_passes = passes_raw[passes_raw['success'] == 1]
unsuccessful_passes = passes_raw[passes_raw['success'] == 0]

print(f"\nDataset sizes:")
print(f"  Successful passes:   {len(successful_passes):,} ({len(successful_passes)/len(passes_raw)*100:.1f}%)")
print(f"  Unsuccessful passes: {len(unsuccessful_passes):,} ({len(unsuccessful_passes)/len(passes_raw)*100:.1f}%)")

print(f"\n{'Variable':<40s} {'Available':<12s} {'Successful %':<15s} {'Unsuccessful %':<15s} {'Status':<10s}")
print("-"*100)

results = []
for var in proposed_vars:
    if var in passes_raw.columns:
        avail = "✅ YES"
        succ_pct = (1 - successful_passes[var].isna().mean()) * 100
        unsucc_pct = (1 - unsuccessful_passes[var].isna().mean()) * 100
        
        # Determine status
        if succ_pct >= 95 and unsucc_pct >= 95:
            status = "✅ GOOD"
        elif unsucc_pct >= 80:
            status = "⚠️ OK"
        else:
            status = "❌ SPARSE"
            
        results.append({
            'variable': var,
            'available': True,
            'succ_pct': succ_pct,
            'unsucc_pct': unsucc_pct,
            'status': status
        })
        
        print(f"{var:<40s} {avail:<12s} {succ_pct:>12.1f}%   {unsucc_pct:>12.1f}%   {status:<10s}")
    else:
        avail = "❌ NO"
        status = "❌ MISSING"
        results.append({
            'variable': var,
            'available': False,
            'succ_pct': 0,
            'unsucc_pct': 0,
            'status': status
        })
        print(f"{var:<40s} {avail:<12s} {'N/A':>12s}   {'N/A':>12s}   {status:<10s}")

# Summary
results_df = pd.DataFrame(results)
available = results_df['available'].sum()
good = (results_df['status'] == '✅ GOOD').sum()
ok = (results_df['status'] == '⚠️ OK').sum()
problematic = len(results_df) - good - ok

print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"Total proposed variables: {len(proposed_vars)}")
print(f"  ✅ Available in data:   {available} ({available/len(proposed_vars)*100:.0f}%)")
print(f"  ❌ Missing from data:   {len(proposed_vars) - available}")
print()
print(f"Data quality (for available variables):")
print(f"  ✅ GOOD (≥95% non-null for both):     {good}")
print(f"  ⚠️ OK (≥80% non-null for unsuccessful): {ok}")
print(f"  ❌ PROBLEMATIC (<80% for unsuccessful): {problematic}")

VARIABLE AVAILABILITY ANALYSIS

Dataset sizes:
  Successful passes:   272,335 (82.8%)
  Unsuccessful passes: 56,378 (17.2%)

Variable                                 Available    Successful %    Unsuccessful %  Status    
----------------------------------------------------------------------------------------------------
x_start                                  ✅ YES               100.0%          100.0%   ✅ GOOD    
y_start                                  ✅ YES               100.0%          100.0%   ✅ GOOD    
x_end                                    ✅ YES               100.0%          100.0%   ✅ GOOD    
y_end                                    ✅ YES               100.0%          100.0%   ✅ GOOD    
third_start                              ✅ YES               100.0%          100.0%   ✅ GOOD    
channel_start                            ✅ YES               100.0%          100.0%   ✅ GOOD    
penalty_area_start                       ✅ YES               100.0%          100.0%   ✅ GOOD   

## 3. Model Selection & Rationale

### Why XGBoost?

**Advantages for this task**:
1. **Handles missing values** naturally (high_pass, speed_avg)
2. **Mixed data types**: Numeric (coordinates, separation) + Categorical (game_state, phases)
3. **Non-linear interactions**: Field position × game context are critical
4. **Feature importance**: Built-in analysis of which variables matter
5. **Robust to outliers**: Common in sports data
6. **Industry standard**: Proven for classification in sports analytics

### Target Variable

**6-class classification**:
- `short_backward`
- `short_lateral`
- `short_forward`
- `long_backward`
- `long_lateral`
- `long_forward`

### Expected Performance

- **Training samples**: ~272K successful passes
- **Expected accuracy**: 70-85%
- **Most confusion**: short_lateral ↔ short_forward (similar contexts)
- **Least confusion**: backward passes (distinct tactical situations)

### Alternative Models Considered

- ❌ **Logistic Regression**: Too simple, can't capture position × context interactions
- ⚠️ **Random Forest**: Good alternative, but XGBoost typically outperforms
- ⚠️ **LightGBM**: Similar to XGBoost, faster but less feature support
- ❌ **Neural Network**: Overkill, requires more data and tuning

## 4. Final Feature Set

Based on availability analysis, we'll use the following features:

In [4]:
# Filter to features with good availability (≥80% for unsuccessful)
good_features = results_df[
    (results_df['available'] == True) & 
    (results_df['unsucc_pct'] >= 80)
]['variable'].tolist()

print("="*80)
print(f"FINAL FEATURE SET: {len(good_features)} features")
print("="*80)
print("\nFeatures to use for model training:")
for i, feat in enumerate(good_features, 1):
    status = results_df[results_df['variable']==feat]['status'].iloc[0]
    unsucc_pct = results_df[results_df['variable']==feat]['unsucc_pct'].iloc[0]
    print(f"  {i:2d}. {feat:<40s} ({unsucc_pct:>5.1f}% available for unsuccessful)")

print(f"\n{'='*80}")
print("NOTES:")
print(f"{'='*80}")
print("""\n- We exclude high_pass (0% for unsuccessful)
- We include speed_avg despite 8% missing (XGBoost handles this)
- All spatial features use START location (where pass was made)
- No geometric features (distance/angle) since we lack reception coords
- Model must learn from tactical context rather than pass trajectory
""")

FINAL FEATURE SET: 20 features

Features to use for model training:
   1. x_start                                  (100.0% available for unsuccessful)
   2. y_start                                  (100.0% available for unsuccessful)
   3. x_end                                    (100.0% available for unsuccessful)
   4. y_end                                    (100.0% available for unsuccessful)
   5. third_start                              (100.0% available for unsuccessful)
   6. channel_start                            (100.0% available for unsuccessful)
   7. penalty_area_start                       (100.0% available for unsuccessful)
   8. game_state                               (100.0% available for unsuccessful)
   9. team_in_possession_phase_type            (100.0% available for unsuccessful)
  10. team_out_of_possession_phase_type        (100.0% available for unsuccessful)
  11. n_teammates_ahead_start                  (100.0% available for unsuccessful)
  12. n_opponents_a

## 5. Next Steps

### Implementation Plan

1. **Feature Engineering**
   - One-hot encode categorical variables (game_state, phase_types)
   - Normalize coordinates to [0, 1] range
   - Handle missing values in speed_avg (median imputation or let XGBoost handle)

2. **Target Variable Preparation**
   - For successful passes: use existing pass_type classification
   - Ensure all 6 classes are represented
   - Check class balance (may need stratified sampling)

3. **Train/Validation Split**
   - 80/20 split on successful passes
   - Stratify by pass_type to maintain class proportions
   - Set random seed for reproducibility

4. **Model Training**
   - Start with default XGBoost hyperparameters
   - Use multi:softmax objective (6 classes)
   - Monitor train/val loss to detect overfitting

5. **Hyperparameter Tuning**
   - max_depth: [3, 5, 7]
   - learning_rate: [0.01, 0.05, 0.1]
   - n_estimators: [100, 300, 500]
   - subsample: [0.8, 1.0]
   - Use cross-validation on training set

6. **Evaluation**
   - Overall accuracy
   - Per-class precision, recall, F1
   - Confusion matrix
   - Feature importance analysis

7. **Application to Unsuccessful Passes**
   - Predict pass_type for ~56K unsuccessful passes
   - Compare distribution to successful passes
   - Analyze prediction confidence

8. **MDP Integration**
   - Use predicted pass_types to assign action IDs
   - Construct complete transition matrix
   - Validate that MDP constraints are satisfied

---

**Ready to proceed with implementation!**

## 6. Feature Engineering

### Step 1: Filter to 100% Available Features

Remove `speed_avg` and `high_pass` to keep only features with 100% availability for unsuccessful passes.

In [5]:
# Filter to only features with 100% availability for unsuccessful passes
final_features = [
    feat for feat in good_features 
    if results_df[results_df['variable']==feat]['unsucc_pct'].iloc[0] == 100.0
]

# Remove speed_avg and high_pass explicitly (not 100% available)
final_features = [f for f in final_features if f not in ['speed_avg', 'high_pass']]

print("="*80)
print("FINAL FEATURE SET (100% available only)")
print("="*80)
print(f"\nTotal features: {len(final_features)}")
print("\nFeatures:")
for i, feat in enumerate(final_features, 1):
    print(f"  {i:2d}. {feat}")
    
print(f"\n{'='*80}")
print("Dropped features:")
print("  - speed_avg (92.1% available for unsuccessful)")
print("  - high_pass (0% available for unsuccessful)")
print(f"{'='*80}")

FINAL FEATURE SET (100% available only)

Total features: 19

Features:
   1. x_start
   2. y_start
   3. x_end
   4. y_end
   5. third_start
   6. channel_start
   7. penalty_area_start
   8. game_state
   9. team_in_possession_phase_type
  10. team_out_of_possession_phase_type
  11. n_teammates_ahead_start
  12. n_opponents_ahead_start
  13. separation_start
  14. one_touch
  15. quick_pass
  16. carry
  17. forward_momentum
  18. is_header
  19. hand_pass

Dropped features:
  - speed_avg (92.1% available for unsuccessful)
  - high_pass (0% available for unsuccessful)


### Step 2: Identify Feature Types

Categorize features into numeric and categorical for appropriate preprocessing.

In [6]:
# Separate features by type
numeric_features = [
    'x_start', 'y_start', 'x_end', 'y_end',
    'n_teammates_ahead_start',
    'n_opponents_ahead_start',
    'separation_start'
]

categorical_features = [
    'third_start',
    'channel_start',
    'game_state',
    'team_in_possession_phase_type',
    'team_out_of_possession_phase_type'
]

boolean_features = [
    'penalty_area_start',
    'one_touch',
    'quick_pass',
    'carry',
    'forward_momentum',
    'is_header',
    'hand_pass'
]

print("="*80)
print("FEATURE TYPES")
print("="*80)
print(f"\nNumeric features ({len(numeric_features)}):")
for feat in numeric_features:
    print(f"  - {feat}")

print(f"\nCategorical features ({len(categorical_features)}):")
for feat in categorical_features:
    print(f"  - {feat}")
    
print(f"\nBoolean features ({len(boolean_features)}):")
for feat in boolean_features:
    print(f"  - {feat}")

print(f"\nTotal: {len(numeric_features) + len(categorical_features) + len(boolean_features)} features")

# Verify we have all features
assert set(numeric_features + categorical_features + boolean_features) == set(final_features), \
    "Feature type lists don't match final_features!"
print("\n✅ All features categorized correctly")

FEATURE TYPES

Numeric features (7):
  - x_start
  - y_start
  - x_end
  - y_end
  - n_teammates_ahead_start
  - n_opponents_ahead_start
  - separation_start

Categorical features (5):
  - third_start
  - channel_start
  - game_state
  - team_in_possession_phase_type
  - team_out_of_possession_phase_type

Boolean features (7):
  - penalty_area_start
  - one_touch
  - quick_pass
  - carry
  - forward_momentum
  - is_header
  - hand_pass

Total: 19 features

✅ All features categorized correctly


### Step 3: Prepare Data for Training

Extract successful passes with pass_type labels (our training data).

In [7]:
# We need to get pass_type for successful passes
# First, let's rescale and classify successful passes
print("Preparing training data (successful passes with pass_type)...")

# Rescale coordinates for successful passes
successful_rescaled = dp.rescale_coordinates(
    successful_passes,
    x_cols=('x_end', 'player_targeted_x_reception'),
    y_cols=('y_end', 'player_targeted_y_reception')
)

# Normalize attack direction
successful_rescaled = dp.normalize_attack_direction(successful_rescaled)

# Classify pass types
successful_classified = dp.classify_pass_type(successful_rescaled)

print(f"\n{'='*60}")
print(f"SUCCESSFUL PASSES WITH PASS_TYPE")
print(f"{'='*60}")
print(f"Total successful passes: {len(successful_classified):,}")
print(f"\nPass type distribution:")
print(successful_classified['pass_type'].value_counts())

# Check that we have all required features
missing_features = [f for f in final_features if f not in successful_classified.columns]
if missing_features:
    print(f"\n⚠️ WARNING: Missing features: {missing_features}")
else:
    print(f"\n✅ All {len(final_features)} features available in data")

Preparing training data (successful passes with pass_type)...
✅ Attack direction normalized:
   Flipped: 137,985 / 272,335 (50.7%)
   All teams now attack left→right (x: 0→105)
🔧 Filtered zero-distance passes:
   Removed: 0 (0.00%)
   Remaining: 272,335
✅ Pass types classified:
   Length × Direction = 2 × 3 = 6 unique types
   Total action space: 6 passes + shoot + carry = 8 actions

SUCCESSFUL PASSES WITH PASS_TYPE
Total successful passes: 272,335

Pass type distribution:
pass_type
short_lateral     135809
short_forward      50460
short_backward     50109
long_lateral       20024
long_backward       8064
long_forward        7869
Name: count, dtype: int64

✅ All 19 features available in data


### Step 4: One-Hot Encode Categorical Variables

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

# Extract features and target
X = successful_classified[final_features].copy()
y = successful_classified['pass_type'].copy()

print("="*80)
print("ONE-HOT ENCODING CATEGORICAL VARIABLES")
print("="*80)

# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_categorical_encoded = encoder.fit_transform(X[categorical_features])

# Get feature names for encoded features
encoded_feature_names = encoder.get_feature_names_out(categorical_features)

print(f"\nOriginal categorical features: {len(categorical_features)}")
print(f"After one-hot encoding: {len(encoded_feature_names)} features")

print(f"\nEncoding breakdown:")
for feat in categorical_features:
    n_categories = X[feat].nunique()
    print(f"  {feat}: {n_categories} unique values → {n_categories} binary features")

print(f"\nSample encoded feature names:")
for name in encoded_feature_names[:10]:
    print(f"  - {name}")
if len(encoded_feature_names) > 10:
    print(f"  ... and {len(encoded_feature_names) - 10} more")

print(f"\n✅ Categorical encoding complete")

ONE-HOT ENCODING CATEGORICAL VARIABLES

Original categorical features: 5
After one-hot encoding: 29 features

Encoding breakdown:
  third_start: 3 unique values → 3 binary features
  channel_start: 5 unique values → 5 binary features
  game_state: 3 unique values → 3 binary features
  team_in_possession_phase_type: 9 unique values → 9 binary features
  team_out_of_possession_phase_type: 9 unique values → 9 binary features

Sample encoded feature names:
  - third_start_attacking_third
  - third_start_defensive_third
  - third_start_middle_third
  - channel_start_center
  - channel_start_half_space_left
  - channel_start_half_space_right
  - channel_start_wide_left
  - channel_start_wide_right
  - game_state_drawing
  - game_state_losing
  ... and 19 more

✅ Categorical encoding complete


### Step 5: Normalize Numeric Features

In [9]:
print("="*80)
print("NORMALIZING NUMERIC FEATURES")
print("="*80)

# Normalize numeric features using StandardScaler (mean=0, std=1)
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X[numeric_features])

print(f"\nNumeric features normalized: {len(numeric_features)}")
print("\nOriginal ranges:")
for i, feat in enumerate(numeric_features):
    orig_min = X[feat].min()
    orig_max = X[feat].max()
    orig_mean = X[feat].mean()
    print(f"  {feat:<30s}: [{orig_min:8.2f}, {orig_max:8.2f}], mean={orig_mean:8.2f}")

print("\nScaled statistics (should be mean≈0, std≈1):")
for i, feat in enumerate(numeric_features):
    scaled_mean = X_numeric_scaled[:, i].mean()
    scaled_std = X_numeric_scaled[:, i].std()
    print(f"  {feat:<30s}: mean={scaled_mean:7.4f}, std={scaled_std:7.4f}")

print(f"\n✅ Numeric normalization complete")

NORMALIZING NUMERIC FEATURES

Numeric features normalized: 7

Original ranges:
  x_start                       : [  -56.01,    57.33], mean=   -4.07
  y_start                       : [  -43.51,    43.70], mean=    0.84
  x_end                         : [  -56.79,    57.33], mean=   -2.66
  y_end                         : [  -43.51,    44.67], mean=    0.79
  n_teammates_ahead_start       : [    0.00,    10.00], mean=    5.94
  n_opponents_ahead_start       : [    0.00,    10.00], mean=    8.06
  separation_start              : [    0.01,    56.31], mean=    7.04

Scaled statistics (should be mean≈0, std≈1):
  x_start                       : mean=-0.0000, std= 1.0000
  y_start                       : mean= 0.0000, std= 1.0000
  x_end                         : mean=-0.0000, std= 1.0000
  y_end                         : mean= 0.0000, std= 1.0000
  n_teammates_ahead_start       : mean= 0.0000, std= 1.0000
  n_opponents_ahead_start       : mean= 0.0000, std= 1.0000
  separation_start       

### Step 6: Combine All Features

Combine normalized numeric, encoded categorical, and boolean features into final feature matrix.

In [11]:
print("="*80)
print("COMBINING ALL FEATURES")
print("="*80)

# Extract boolean features (already 0/1)
X_boolean = X[boolean_features].values

# Combine all features: numeric (scaled) + categorical (encoded) + boolean
X_final = np.hstack([
    X_numeric_scaled,      # 7 features
    X_categorical_encoded, # 29 features
    X_boolean             # 7 features
])

# Create feature names list for reference
final_feature_names = (
    numeric_features + 
    list(encoded_feature_names) + 
    boolean_features
)

print(f"\nFeature matrix shape: {X_final.shape}")
print(f"  Samples: {X_final.shape[0]:,}")
print(f"  Features: {X_final.shape[1]}")

print(f"\nFeature breakdown:")
print(f"  Numeric (normalized):       {len(numeric_features)}")
print(f"  Categorical (one-hot):      {len(encoded_feature_names)}")
print(f"  Boolean (0/1):              {len(boolean_features)}")
print(f"  Total:                      {len(final_feature_names)}")

# Ensure X_final is float type for NaN/Inf checks
X_final = X_final.astype(np.float64)

# Verify no NaN or inf values
n_nan = np.isnan(X_final).sum()
n_inf = np.isinf(X_final).sum()
print(f"\nData quality check:")
print(f"  NaN values: {n_nan}")
print(f"  Inf values: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print(f"  ❌ WARNING: Found {n_nan} NaN and {n_inf} Inf values!")
else:
    print(f"  ✅ No NaN or Inf values")

# Check target variable
print(f"\nTarget variable (pass_type):")
print(f"  Unique classes: {y.nunique()}")
print(f"  Distribution:")
for ptype, count in y.value_counts().items():
    print(f"    {ptype}: {count:,} ({count/len(y)*100:.1f}%)")

print(f"\n✅ Feature engineering complete!")
print(f"Ready for train/test split and model training")

COMBINING ALL FEATURES

Feature matrix shape: (272335, 43)
  Samples: 272,335
  Features: 43

Feature breakdown:
  Numeric (normalized):       7
  Categorical (one-hot):      29
  Boolean (0/1):              7
  Total:                      43

Data quality check:
  NaN values: 0
  Inf values: 0
  ✅ No NaN or Inf values

Target variable (pass_type):
  Unique classes: 6
  Distribution:
    short_lateral: 135,809 (49.9%)
    short_forward: 50,460 (18.5%)
    short_backward: 50,109 (18.4%)
    long_lateral: 20,024 (7.4%)
    long_backward: 8,064 (3.0%)
    long_forward: 7,869 (2.9%)

✅ Feature engineering complete!
Ready for train/test split and model training


### Feature Engineering Summary

**Final Dataset**:
- **Training samples**: 272,335 successful passes
- **Features**: 43 total
  - 7 numeric (normalized with StandardScaler)
  - 29 categorical (one-hot encoded from 5 original features)
  - 7 boolean (kept as 0/1)
- **Target**: 6 pass type classes (short/long × backward/lateral/forward)
- **Data quality**: ✅ No missing values, no NaN/Inf

**Class Distribution**:
- Highly imbalanced: `short_lateral` dominates at 49.9%
- Long passes are rare: `long_forward` only 2.9%
- Will need stratified sampling for train/val split

**Next Steps**:
1. Train/validation split (80/20, stratified)
2. XGBoost model training
3. Evaluate performance
4. Apply to unsuccessful passes